# nb25 - GravNet on all cells + photon-time auxiliary task

Two new levers, both using only data we already have:

1. **GravNet** (arXiv:1902.07987 - dynamic graph, O(N*k) message passing in a learned latent space) lets us use **all cells** of the cluster instead of a kNN window. The window scan proved more cells help (kNN-25 0.0483 -> kNN-81 0.0459) but attention is O(N^2); GravNet pays O(N*k), so full containment becomes affordable.
2. **Auxiliary photon-time target**: `sig_flux_timing` (true photon arrival time) has never been used. A small aux head predicting it from the cell set forces the representation to learn the cluster's time structure - richer supervision than the (neutral) in-time gate. Note: a 'pileup fraction' aux was considered and rejected - it is a reparametrisation of the main residual target.

Baselines to beat: GateHuber kNN-25 0.0483 (3 seeds), kNN-81 0.0459, seed-ensemble 0.0463.

In [1]:
import os, sys, pathlib, copy, time
import numpy as np, pandas as pd, uproot, awkward as ak, matplotlib.pyplot as plt
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS, CELL_KEYS
FILES = sorted((REPO / 'data' / 'minimum_bias').glob('matched_*.root'))
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
FIG = REPO / 'reports' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = os.environ.get('NB25_MODE', 'full')
print('device', DEVICE, '| mode', MODE, '|', len(FILES), 'files')

device cuda | mode full | 94 files


## Build: ALL cells per cluster (sorted by distance to seed, capped)
No kNN selection. Cells sorted by distance to seed and capped at `LCAP` (covers the vast majority of clusters fully - coverage printed). Per-cell 15-dim tokens as in nb16-22 (log energies, geometry, pitch one-hot, dtf/dtb/hasv). Globals include the seed absolute time so the aux head can relate cell times to the photon time.

In [2]:
TKEYS = CELL_KEYS + ['cell_times_front', 'cell_times_back']
AUX = ['sig_flux_prod_vertex_z', 'sig_flux_eTot', 'sig_flux_timing']
LCAP = 240
def build_all(files, vertex_max=100.0):
    O = {k: [] for k in ['tok', 'agg', 'y', 'Etrue', 'ptime', 't0']}
    nfull = 0; ntot = 0
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TKEYS + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        pt = ak.to_numpy(a['sig_flux_timing']).astype(float)
        for i in np.flatnonzero(vz < vertex_max):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TKEYS}
            e = cc['energy']
            if len(e) == 0: continue
            seed = int(np.argmax(e))
            x = cc['cell_x']; yy = cc['cell_y']; ix = cc['imodx']; iy = cc['jmody']
            pts = np.stack([x, yy], 1)
            pitch = np.full(len(x), np.nan)
            for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
                sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
                if len(p) >= 2:
                    d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
                    pitch[sel] = np.median(np.min(d, axis=1))
            fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
            pitch[~np.isfinite(pitch)] = fill
            mod = np.array([int(np.argmin(np.abs(PITCH - p))) for p in pitch], dtype=int)
            rx = x - x[seed]; ry = yy - yy[seed]; rdr = np.hypot(rx, ry)
            o = np.argsort(rdr); ntot += 1; nfull += int(len(e) <= LCAP)
            o = o[:LCAP]
            e2, fr, bk = e[o], cc['cell_energies_front'][o], cc['cell_energies_back'][o]
            tf, tb = cc['cell_times_front'][o], cc['cell_times_back'][o]
            rx, ry, rdr, pit, md_ = rx[o], ry[o], rdr[o], pitch[o], mod[o]
            vf = np.abs(tf) < 1e6; vb = np.abs(tb) < 1e6
            t0f = tf[0] if vf[0] else (np.median(tf[vf]) if vf.any() else 0.0)
            t0b = tb[0] if vb[0] else (np.median(tb[vb]) if vb.any() else 0.0)
            dtf = np.where(vf, tf - t0f, 0.0); dtb = np.where(vb, tb - t0b, 0.0)
            hasv = (vf | vb).astype(float)
            cont = np.stack([np.log1p(np.clip(e2, 0, None)), np.log1p(np.clip(fr, 0, None)),
                             np.log1p(np.clip(bk, 0, None)), rx / pit, ry / pit, rdr / pit,
                             np.log(pit), dtf, dtb], 1)
            oh = np.zeros((len(e2), len(PITCH))); oh[np.arange(len(e2)), md_] = 1.0
            O['tok'].append(np.concatenate([cont, hasv[:, None], oh], 1).astype(np.float32))
            sumE = float(e2.sum()); seedE = float(e2[0])
            lat = float(np.sqrt((e2 * rdr ** 2).sum() / (sumE + EPS)))
            fb = float(fr.sum() / (bk.sum() + EPS))
            O['agg'].append([np.log1p(sumE), fb, len(e2), np.log1p(seedE), lat, int(md_[0])])
            et = float(a['sig_flux_eTot'][i])
            O['y'].append(np.log(max(et, 1e-3))); O['Etrue'].append(et)
            O['ptime'].append(float(pt[i])); O['t0'].append(float(t0f))
    for k in ['agg', 'y', 'Etrue', 'ptime', 't0']: O[k] = np.array(O[k])
    print(f'clusters {ntot} | fully contained at LCAP={LCAP}: {100*nfull/max(ntot,1):.1f}%')
    return O

## GravNet model
Blocks: per-cell MLP -> latent coords S (4-dim) + features F; kNN (k=12) in S-space; neighbours aggregated with a Gaussian distance potential (mean+max); residual update. Pooling = energy-weighted sum (EFN) of the final features + globals + calibrated-sum base -> residual energy head (Huber). Aux head predicts the normalised photon time from the pooled representation (weight `lam`).

In [3]:
CFG = dict(d=64, ds=4, dlr=32, k=12, blocks=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1)
N_GLOBAL = 7
class GravBlock(nn.Module):
    def __init__(self, d, ds, dlr, k, drop):
        super().__init__(); self.k = k
        self.pre = nn.Sequential(nn.Linear(d, d), nn.GELU())
        self.s = nn.Linear(d, ds); self.f = nn.Linear(d, dlr)
        self.out = nn.Sequential(nn.Linear(d + 2 * dlr, d), nn.GELU(), nn.Dropout(drop))
        self.norm = nn.LayerNorm(d)
    def forward(self, h, m):
        x = self.pre(h)
        S = self.s(x); F = self.f(x)
        d2 = torch.cdist(S, S) ** 2
        d2 = d2.masked_fill(~m.unsqueeze(1), 1e9)
        k = min(self.k, d2.shape[-1])
        dk, idx = torch.topk(d2, k, dim=-1, largest=False)
        w = torch.exp(-10.0 * dk).unsqueeze(-1)
        B, L, _ = F.shape
        nb = torch.gather(F.unsqueeze(1).expand(B, L, L, -1), 2,
                          idx.unsqueeze(-1).expand(B, L, k, F.shape[-1]))
        agg_mean = (nb * w).mean(2); agg_max = (nb * w).max(2).values
        upd = self.out(torch.cat([x, agg_mean, agg_max], -1))
        return self.norm(h + upd) * m.unsqueeze(-1).float()
class GravNet(nn.Module):
    def __init__(self, in_dim, aux_time=True):
        super().__init__(); d = CFG['d']; self.aux_time = aux_time
        self.embed = nn.Linear(in_dim, d)
        self.blocks = nn.ModuleList([GravBlock(d, CFG['ds'], CFG['dlr'], CFG['k'], CFG['dropout'])
                                     for _ in range(CFG['blocks'])])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + N_GLOBAL, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        if aux_time:
            self.thead = nn.Sequential(nn.Linear(d + N_GLOBAL, 32), nn.GELU(), nn.Linear(32, 1))
    def forward(self, x, m, w, g, base):
        h = self.embed(x) * m.unsqueeze(-1).float()
        for blk in self.blocks: h = blk(h, m)
        p = self.norm((h * w.unsqueeze(-1)).sum(1))
        z = torch.cat([p, g], 1)
        e = base + self.head(z)
        t = self.thead(z) if self.aux_time else None
        return e, t

## Prep tensors (CPU-resident, batches moved to GPU)

In [4]:
O = build_all(FILES if MODE == 'full' else FILES[:8])
N = len(O['Etrue']); Et = O['Etrue']; y = O['y'].astype(np.float32); agg = O['agg']
keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
ktr, kva, kte = (keep[s] for s in split(len(keep)))
tmed = np.median(O['ptime']); tscale = np.subtract(*np.percentile(O['ptime'], [75, 25])) / 1.349 + EPS
ptn = ((O['ptime'] - tmed) / tscale).astype(np.float32)
t0n = ((O['t0'] - np.median(O['t0'])) / (np.subtract(*np.percentile(O['t0'], [75, 25])) / 1.349 + EPS)).astype(np.float32)
G = np.stack([agg[:,0], agg[:,3], np.log(agg[:,2]+1.0), agg[:,1], agg[:,4],
              t0n, np.clip(agg[:,2], 0, LCAP)/LCAP], 1).astype(np.float32)
G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
la, lb = np.polyfit(agg[ktr,0], y[ktr], 1); base_all = (la*agg[:,0] + lb).astype(np.float32)
maxL = max(t.shape[0] for t in O['tok'])
IN_DIM = O['tok'][0].shape[1]
X = np.zeros((N, maxL, IN_DIM), np.float32); M = np.zeros((N, maxL), np.bool_); W = np.zeros((N, maxL), np.float32)
for i, t in enumerate(O['tok']):
    L = t.shape[0]; X[i, :L] = t; M[i, :L] = True
    e = np.expm1(np.clip(t[:, 0], 0, None)); W[i, :L] = e / (e.sum() + 1e-9)
cont = X[ktr][:, :, :9].reshape(-1, 9)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :9] = (X[:, :, :9] - mean) / std; X[~M] = 0.0
Xc = torch.from_numpy(X).to(DEVICE); Mc = torch.from_numpy(M).to(DEVICE); Wc = torch.from_numpy(W).to(DEVICE)
Gc = torch.from_numpy(G).to(DEVICE); Bc = torch.from_numpy(base_all).unsqueeze(1).to(DEVICE)
Yc = torch.from_numpy(y).unsqueeze(1).to(DEVICE); Tc = torch.from_numpy(ptn).unsqueeze(1).to(DEVICE)
print('N', N, 'maxL', maxL, 'in_dim', IN_DIM, 'train/val/test', len(ktr), len(kva), len(kte))

clusters 89797 | fully contained at LCAP=240: 80.4%


N 89797 maxL 240 in_dim 15 train/val/test 57056 12226 12227


## Train / eval

In [5]:
EPOCHS = {'smoke': 3, 'full': 120}[MODE]
PATIENCE = {'smoke': 99, 'full': 20}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1, 2]}[MODE]
def train_eval(aux_lam, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = GravNet(IN_DIM, aux_time=aux_lam > 0).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    def fwd(b):
        bb = b.to(DEVICE)
        return model(Xc[b].to(DEVICE), Mc[b].to(DEVICE), Wc[b].to(DEVICE), Gc[b].to(DEVICE), Bc[b].to(DEVICE)), bb
    def lossf(out, b):
        (e, t), bb = out
        L = nn.functional.huber_loss(e, Yc[b].to(DEVICE), delta=CFG['huber_delta'])
        if t is not None:
            L = L + aux_lam * nn.functional.huber_loss(t, Tc[b].to(DEVICE), delta=1.0)
        return L
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs])
    def run(idx):
        out = []
        model.eval()
        with torch.no_grad():
            for b in batches(idx, 384, False):
                (e, _), _ = fwd(b); out.append(e.cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; n = 0
        with torch.no_grad():
            for b in batches(kva, 384, False): s += lossf(fwd(b), b).item(); n += 1
        return s / max(n, 1)
    best = 1e9; bstate = None; wait = 0
    for ep in range(EPOCHS):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad(); lossf(fwd(b), b).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(bstate)
    a, b2 = np.polyfit(run(kva), y[kva], 1)
    pe = np.exp(a * run(kte) + b2)
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

## Run: aux on (3 seeds) + aux off (1 seed isolation) - resumable

In [6]:
CSVP = OUT / 'nb25_gravnet.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
PREDS = {}
todo = [('gravnet_aux', 0.1, s) for s in SEEDS] + ([('gravnet_noaux', 0.0, 0)] if MODE == 'full' else [])
for name, lam, seed in todo:
    if (name, seed) in done:
        print('skip', name, seed); continue
    t0 = time.time()
    sig, pe = train_eval(lam, seed)
    PREDS[(name, seed)] = pe
    row = dict(config=name, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t0))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{name} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
RES = pd.read_csv(CSVP); print(RES.to_string(index=False))

gravnet_aux seed 0: sigma_eff 0.0518 (5311s)


gravnet_aux seed 1: sigma_eff 0.0528 (5926s)


gravnet_aux seed 2: sigma_eff 0.0521 (5119s)


gravnet_noaux seed 0: sigma_eff 0.0562 (3369s)


       config  seed  sigma_eff  elapsed
  gravnet_aux     0     0.0518     5311
  gravnet_aux     1     0.0528     5926
  gravnet_aux     2     0.0521     5119
gravnet_noaux     0     0.0562     3369


In [7]:
aux = RES[RES.config == 'gravnet_aux']
print(f'GravNet all-cell + aux-time: {aux.sigma_eff.mean():.4f} +/- {aux.sigma_eff.std():.4f} ({len(aux)} seeds)')
if len(PREDS) >= 2:
    stack = np.stack([p for (n, s), p in PREDS.items() if n == 'gravnet_aux'])
    if len(stack) >= 2:
        print('seed-ensemble:', resolution(stack.mean(0), Et[kte])['sigma_eff'])
print('baselines: GateHuber kNN-25 0.0483 | kNN-81 0.0459 | kNN-25 seed-ensemble 0.0463')

GravNet all-cell + aux-time: 0.0522 +/- 0.0005 (3 seeds)
seed-ensemble: 0.0495
baselines: GateHuber kNN-25 0.0483 | kNN-81 0.0459 | kNN-25 seed-ensemble 0.0463
